## Great Lakes Dashboard

In [4]:
import pandas as pd
import plotly.express as px

In [9]:
# load in csv

joyce_data = pd.read_csv('../data/R Results_Updated(Indirect Water Use Table).csv')

In [23]:
#print columns
#take away spaces
joyce_data.columns = joyce_data.columns.str.strip()
joyce_data["State"] = joyce_data["State"].str.strip()
print(joyce_data.columns)

Index(['Scenario', 'State', 'Generation 2025 (MWh)', 'Generation 2030 (MWh)',
       'Change in Generation (MWh)', 'Indirect Water Use 2025 (gallons)',
       'Indirect Water Use 2030 (gallons)',
       'Change in Indirect Water Use (gallons)',
       'Indirect Water Use 2025 (liters)', 'Indirect Water Use 2030 (liters)',
       'Change in Indirect Water Use (liters)',
       'Direction of Water Use Change'],
      dtype='object')


### Regional Overview
**Total Data Center Electricity Demand**

In [27]:
# turn the number columns from text like ' 31,290,777.33 ' into real numbers
num_cols = joyce_data.columns.drop(["State", "Direction of Water Use Change"])
for col in num_cols:
    joyce_data[col] = pd.to_numeric(
        joyce_data[col].astype(str)
            .str.replace(",", "")      # 31,290,777 -> 31290777
            .str.replace("(", "-")     # (2578460304.84) -> -2578460304.84)
            .str.replace(")", "")      # -2578460304.84) -> -2578460304.84
            .str.strip()               # remove the extra spaces
    )

joyce_data.dtypes

Scenario                                    int64
State                                      object
Generation 2025 (MWh)                     float64
Generation 2030 (MWh)                     float64
Change in Generation (MWh)                float64
Indirect Water Use 2025 (gallons)         float64
Indirect Water Use 2030 (gallons)         float64
Change in Indirect Water Use (gallons)    float64
Indirect Water Use 2025 (liters)          float64
Indirect Water Use 2030 (liters)          float64
Change in Indirect Water Use (liters)     float64
Direction of Water Use Change              object
dtype: object

In [30]:
# total data center electricity demand in the Great Lakes Region in 2025 and 2030
totals = joyce_data[joyce_data["State"] == "Total"]
row = totals.iloc[0] #same in every scenario so any row works

demand = pd.DataFrame({
    "Year": ["2025", "2030"],
    "Demand (TWh)": [row["Generation 2025 (MWh)"]/1e6, row["Generation 2030 (MWh)"]/1e6],
})

fig_demand = px.bar(demand, x="Year", y="Demand (TWh)", title="Total Data Center Electricity Demand in the Great Lakes Region", text_auto=".1f")

fig_demand.show()
fig_demand.write_html("electricity_demand.html", include_plotlyjs="cdn")  # saves it as a file

**Total Indirect Water Use Each Year**

In [16]:
totals = joyce_data[joyce_data["State"] == "Total"]
print(totals.head)

<bound method NDFrame.head of Empty DataFrame
Columns: [Scenario, State, Generation 2025 (MWh), Generation 2030 (MWh), Change in Generation (MWh), Indirect Water Use 2025 (gallons), Indirect Water Use 2030 (gallons), Change in Indirect Water Use (gallons), Indirect Water Use 2025 (liters), Indirect Water Use 2030 (liters), Change in Indirect Water Use (liters), Direction of Water Use Change]
Index: []>


In [ ]:

# total indirect water use in each year as a range from across scenarios

import plotly.graph_objects as go

scenarios = {1: "High Demand", 2: "Same energy mix", 3: "Mid case", 4: "High renewable cost", 5: "Low renewable cost", 6: "Low gas price", 7: "Low RE cost + high gas", 8: "High RE cost + low gas"}

fig_water = go.Figure()
scenarios = totals["Scenario"].tolist()

#one set of bars per scenario, and show only the first one at start
for i, s in enumerate(scenarios):
    row = totals[totals["Scenario"] == s].iloc[0]
    fig_water.add_trace(go.Bar(
        x=["2025", "2030"],
        y=[row["Indirect Water Use 2025 (gallons)"] / 1e9,
           row["Indirect Water Use 2030 (gallons)"] / 1e9],
        name=scenarios[s],
        visible=(i == 0)  # Show only the first scenario initially

    ))
buttons = []
for i, s in enumerate(scenarios):
    visible = [j == i for j in range(len(scenarios))]
    buttons.append(dict(label=scenarios[s], method="update", args=[{"visible": visible}]))
    
fig_water.update_layout(
    title="Great Lakes indirect water use",
    yaxis_title="Billion gallons",
    updatemenus=[dict(buttons=buttons, x=0, xanchor="left", y=1.15, yanchor="top")],
)

fig_water.show()
fig_water.write_html("regional_water.html", include_plotlyjs="cdn")

IndexError: list index out of range

**Overall Change from 2025 to 2030 for Electricity and Indirect Water Use**

In [29]:
# demand change (the same for every scenario)
d25, d30 = demand["Demand (TWh)"]
print(f"Demand: {d25:.1f} -> {d30:.1f} TWh, +{d30 - d25:.1f} TWh (+{(d30 - d25) / d25 * 100:.0f}%)")

# water change (different for each scenario, so report the range)
w25 = totals["Indirect Water Use 2025 (gallons)"] / 1e9
w30 = totals["Indirect Water Use 2030 (gallons)"] / 1e9
change = w30 - w25
pct = change / w25 * 100
print(f"Water use change: +{change.min():.1f} to +{change.max():.1f} billion gal "
      f"(+{pct.min():.0f}% to +{pct.max():.0f}%)")

Demand: 31.3 -> 138.6 TWh, +107.3 TWh (+343%)
Water use change: +55.4 to +91.9 billion gal (+193% to +321%)
